![Cabecera](../../../Sprint_11/assets/cabecera_thebridge.png)

# Intro - Grafos con LangGraph

Vamos a empezar a trabajar con LangGraph. Empezaremos por entender cómo funcionan conceptos básicos como:

- `StateGraph` y estado compartido (`TypedDict`)
- **Nodos** (funciones que actualizan el estado)
- **Edges** fijas
- `compile` + `invoke`

> Esto **no sustituye** el agente con tools en Python que hemos estado viendo anteoriormente. Es el mismo tipo de flujo dibujado con LangGraph. Más adelante migraremos el agente completo al grafo de LangGraph.

En estos notebooks el LLM se usa vía `langchain-google-genai` (wrapper LangChain + Gemini). La API key es la misma `GEMINI_API_KEY`.


## 1. Dependencias


In [ ]:
# %pip install -qU langgraph langchain-google-genai grandalf python-dotenv


## 2. API key


In [1]:
import os
from getpass import getpass

from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Pega tu GEMINI_API_KEY: ")

print("GEMINI_API_KEY:", "sí" if os.getenv("GEMINI_API_KEY") else "no")


GEMINI_API_KEY: sí


## 3. Qué vamos a construir

En este notebook montamos un **flujo lineal** con LangGraph: el mismo pedido del usuario pasa por dos pasos, uno detrás de otro. Diseñamos el grafo dirigido con LanGraph. El estado se comparte entre los nodos.

```text
De manera genérica:
[START] → [nodo_1] → [nodo_2] → [END]

En nuestro caso:
[START] → [normalizar pedido] → [proponer tarde] → [END]
```

1. **Normalizar** — limpiamos el texto del usuario (espacios, signos).
2. **Proponer** — Gemini escribe un borrador corto de tarde cultural.

Tres piezas de LangGraph que verás en el código:

| Pieza | Qué es | En este ejemplo |
|-------|--------|------------------|
| **Estado** | Un diccionario tipado (`TypedDict`) con lo que “recuerda” el flujo | `pedido`, `pedido_limpio`, `borrador`, `listo` |
| **Nodo** | Una función Python: recibe el estado y **devuelve solo los campos que cambia** | `nodo_normalizar`, `nodo_proponer` |
| **Edge** | La flecha que une un nodo con el siguiente (aquí, fijas: siempre A → B) | START → normalizar → proponer → END |

Al final **compilamos** el grafo (`compile`) y lo **ejecutamos** con un pedido de ejemplo (`invoke`).

No usamos tools todavía: solo texto + LLM. El objetivo es ver cómo se orquesta un flujo con estado compartido.


## 4. Definición del Estado


In [2]:
from typing import TypedDict

# TypedDict = "plantilla" del estado: define qué claves existen y de qué tipo.
# LangGraph pasa este diccionario de nodo en nodo (estado compartido).


class EstadoAgente(TypedDict):
    pedido: str          # texto original que escribe el usuario
    pedido_limpio: str   # misma idea, ya normalizado (lo rellena el nodo normalizar)
    borrador: str        # respuesta corta del LLM (lo rellena el nodo proponer)
    listo: bool          # True cuando ya tenemos borrador (fin del flujo en este ejemplo)


# Muestra solo los nombres y tipos de los campos (útil para comprobar el contrato)
print(EstadoAgente.__annotations__)


{'pedido': <class 'str'>, 'pedido_limpio': <class 'str'>, 'borrador': <class 'str'>, 'listo': <class 'bool'>}


## 5. Modelo (Gemini vía LangChain)


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.3,
)
print("Modelo:", llm.model)


Modelo: gemini-3.1-flash-lite


## 6. Nodos

1. `normalizar` — limpia el pedido.
2. `proponer` — Gemini sugiere un borrador de tarde (sin tools; solo texto).

Los nodos son fuciones que realizan las siguientes tareas:
- Reciben el estado actual
- Realizan una tarea
- Devuelven el estado actualizado

In [9]:
# Un NODO = una función Python.
# Recibe el estado actual y DEVUELVE solo los campos que quiere cambiar
# (LangGraph los fusiona con el resto del estado).


def texto_del_llm(mensaje) -> str:
    """Gemini a veces devuelve .content como str y a veces como list."""
    content = mensaje.content
    if isinstance(content, list):
        partes = []
        for bloque in content:
            if isinstance(bloque, str):
                partes.append(bloque)
            elif isinstance(bloque, dict):
                partes.append(str(bloque.get("text", "")))
            else:
                partes.append(str(getattr(bloque, "text", bloque)))
        return "".join(partes).strip()
    return (content or "").strip()


def nodo_normalizar(estado: EstadoAgente) -> dict:
    """Paso 1: limpiar el texto del usuario (sin llamar al LLM)."""
    # Leemos el pedido original (si falta, usamos cadena vacía)
    pedido = (estado.get("pedido") or "").strip()

    # Quitamos signos finales raros y dejamos un punto al final (texto más limpio)
    sin_signos = pedido.rstrip("?!. ")
    limpio = f"{sin_signos}." if sin_signos else ""

    print(f"[normalizar] '{pedido}' → '{limpio}'")

    # Solo actualizamos estos dos campos; el resto del estado se mantiene
    return {
        "pedido_limpio": limpio,
        "listo": False,  # todavía no hay borrador del LLM
    }


def nodo_proponer(estado: EstadoAgente) -> dict:
    """Paso 2: pedir a Gemini un borrador corto de tarde cultural."""
    # Instrucciones + el pedido ya limpio del nodo anterior
    prompt = (
        "Eres un planificador de tardes culturales en Madrid. "
        "Responde en 2-4 frases concretas (zona, tipo de plan, tono). "
        "No inventes precios exactos ni horarios oficiales.\n\n"
        f"Pedido del usuario: {estado['pedido_limpio']}"
    )

    print("[proponer] llamando al LLM…")
    out = llm.invoke(prompt)  # llamada a Gemini (vía LangChain)
    texto = texto_del_llm(out)  # .content puede ser str o list

    print("[proponer] borrador:", texto[:120], "…")

    return {
        "borrador": texto,  # guardamos la respuesta en el estado
        "listo": True,  # el flujo puede terminar
    }


print("Nodos listos: normalizar, proponer")


Nodos listos: normalizar, proponer


## 7. Contrucción del Grafo lineal


En esta parte se unen los Nodos y Edges para generar el grafo de LangGraph

In [10]:
from langgraph.graph import StateGraph, START, END

grafo = StateGraph(EstadoAgente)
grafo.add_node("normalizar", nodo_normalizar)
grafo.add_node("proponer", nodo_proponer)

grafo.add_edge(START, "normalizar")
grafo.add_edge("normalizar", "proponer")
grafo.add_edge("proponer", END)

app = grafo.compile()
print("Grafo compilado")
print(app.get_graph().draw_ascii())


Grafo compilado
+-----------+  
| __start__ |  
+-----------+  
       *       
       *       
       *       
+------------+ 
| normalizar | 
+------------+ 
       *       
       *       
       *       
 +----------+  
 | proponer |  
 +----------+  
       *       
       *       
       *       
  +---------+  
  | __end__ |  
  +---------+  


## 8. Ejecutar


Cuando vamos a ejecutar, sólo necesitamos rellenar en el estado inicial la pregunta al agente. El resto de campos se rellenarán más adelante

In [11]:
estado_inicial: EstadoAgente = {
    "pedido": "  quiero una tarde cultural barata en el centro  ",
    "pedido_limpio": "",
    "borrador": "",
    "listo": False,
}

final = app.invoke(estado_inicial)
print("--- resultado ---")
print("pedido_limpio:", final["pedido_limpio"])
print("listo:", final["listo"])
print("borrador:\n", final["borrador"])


[normalizar] 'quiero una tarde cultural barata en el centro' → 'quiero una tarde cultural barata en el centro.'
[proponer] llamando al LLM…
[proponer] borrador: Te recomiendo pasear por el Barrio de las Letras para disfrutar de su arquitectura histórica y visitar las exposiciones  …
--- resultado ---
pedido_limpio: quiero una tarde cultural barata en el centro.
listo: True
borrador:
 Te recomiendo pasear por el Barrio de las Letras para disfrutar de su arquitectura histórica y visitar las exposiciones gratuitas de la sede de CaixaForum en el Paseo del Prado. Es un plan perfecto para empaparte de arte y literatura sin gastar apenas nada, moviéndote a pie por el corazón cultural de la ciudad.


## 9. Ejecutar nodo por nodo (`stream`)

Con `invoke` solo ves el **resultado final**.  
Con `stream` LangGraph te va devolviendo el estado **después de cada nodo**: así entiendes el orden `normalizar` → `proponer`.

Cada `paso` es un diccionario `{nombre_del_nodo: campos_que_actualizó}`.

In [12]:
# Otro pedido de prueba (distinto al de la sección 8)
estado_stream = {
    "pedido": "  cine o música en vivo, presupuesto bajo, zona Lavapiés  ",
    "pedido_limpio": "",
    "borrador": "",
    "listo": False,
}

print("Ejecutando con stream (nodo a nodo)...")
print("=" * 60)

for paso in app.stream(estado_stream):
    # paso = { "nombre_del_nodo": {campos actualizados en ese nodo} }
    for nombre_nodo, actualizacion in paso.items():
        print(f"\nNodo ejecutado: '{nombre_nodo}'")
        print(f"  Actualización: {actualizacion}")

print("\n" + "=" * 60)
print("Fin del stream (ya pasaron normalizar y proponer).")

Ejecutando con stream (nodo a nodo)...
[normalizar] 'cine o música en vivo, presupuesto bajo, zona Lavapiés' → 'cine o música en vivo, presupuesto bajo, zona Lavapiés.'

Nodo ejecutado: 'normalizar'
  Actualización: {'pedido_limpio': 'cine o música en vivo, presupuesto bajo, zona Lavapiés.', 'listo': False}
[proponer] llamando al LLM…
[proponer] borrador: Para una tarde con encanto en Lavapiés, te sugiero disfrutar de una sesión de cine independiente en la Cineteca de Matad …

Nodo ejecutado: 'proponer'
  Actualización: {'borrador': 'Para una tarde con encanto en Lavapiés, te sugiero disfrutar de una sesión de cine independiente en la Cineteca de Matadero o buscar música en vivo en los pequeños locales de jazz y fusión de la zona. Es un plan perfecto para sumergirte en el ambiente bohemio del barrio sin gastar mucho, ideal si buscas una experiencia cultural auténtica y relajada.', 'listo': True}

Fin del stream (ya pasaron normalizar y proponer).


## 10. Para llevarnos

| Pieza | Rol |
|-------|-----|
| `TypedDict` | Contrato del estado (como vuestro `AgentState`, más tipado) |
| `StateGraph` | El contenedor del grafo |
| `add_node` | Un paso del flujo. Funciones |
| `add_edge` | Conecta dos nodos en secuencia |
| `compile` | Deja el grafo listo para ejecutar |
| `invoke` | Ejecuta el grafo con un estado inicial y devuelve el estado final |
| `stream` | Ejecuta nodo a nodo y ves cada actualización |


